In [7]:
import pandas as pd
import numpy as np
import json
import ast
import black

%load_ext jupyter_black

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black


In [8]:
raw_data = pd.read_csv("raw_data_2023-06-16_18:39:47.csv", index_col=0)

In [31]:
raw_data

,seller_name,raw_data,dump_time
0,NDR STORE,"{'product_1': {'ean': '0196378567419', 'title'...",1.686934e+09
1,NaN,"{'product_1': {'ean': '', 'title': 'PC Portabl...",1.686934e+09
2,GLD Group,"{'product_1': {'ean': '4711121242069', 'title'...",1.686934e+09
3,TOPBIZ,"{'product_1': {'ean': '0045496453596', 'title'...",1.686934e+09
4,ADMI,"{'product_1': {'ean': '0196378567419', 'title'...",1.686934e+09
5,PCANDCO,"{'product_1': {'ean': '0197192992593', 'title'...",1.686934e+09
6,FRANCE/RENEW,"{'product_1': {'ean': '0196378567419', 'title'...",1.686934e+09
7,ENTER-WEB,"{'product_1': {'ean': '0196119451311', 'title'...",1.686934e+09


In [49]:
all_seller_product = raw_data["raw_data"].values
ast.literal_eval(all_seller_product[1])["product_3"]

{'ean': '0196378567419',
 'title': 'PC Portable LENOVO Ideapad 3 15ALC6 - 15,6" FHD - AMD Ryzen 5-5500U - RAM 8Go - 512Go SSD - AZERTY',
 'seller_price': '518\n€04',
 'seller_name_fp': 'ENTER-WEB',
 'captcha': '',
 'seller_listing': '/mp-228394-len0196378567419.html?Filter=New',
 'product_image': ['https://www.cdiscount.com/pdt2/4/1/9/1/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/2/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/3/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/4/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/5/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3-15alc6-15-6-fhd.jpg',
  'https://www.cdiscount.com/pdt2/4/1/9/6/065x065/len0196378567419/rw/pc-portable-lenovo-ideapad-3

In [17]:
info_all_seller_product = []
product_check = set()
product_without_concurrent = []

for all_product_seller in all_seller_product:
    dict_all_product_seller = ast.literal_eval(all_product_seller)
    for product in dict_all_product_seller.values():
        try:
            info_product = {}
            ean = product["ean"]
            price = product["seller_price"]
            name_seller = product["seller_name_fp"]
            if ean in product_check:
                continue
            # print(name_seller)
            try:
                others_offer = product["offers"]
                pass
            except Exception:
                product_without_concurrent.append(product)
                # print(product["title"])
                # print(product["seller_price"])
                # print("ici", name_seller)
                # print(ean)
                # print("----------------------------")
                continue
            if name_seller == "":
                price_site_seller = price
                all_price_other_seller = {}
                for info_other_product in others_offer:
                    if (
                        info_other_product["seller_name"].lower().replace(" ", "")
                        == "cdiscount"
                    ):
                        continue
                    else:
                        all_price_other_seller[
                            info_other_product["seller_name"]
                        ] = float(
                            info_other_product["seller_price"]
                            .replace("\n", ".")
                            .replace("€", "")
                        )
                info_product["ean"] = ean
                info_product["list_other_price"] = all_price_other_seller
                product_check.add(ean)
                continue
            else:
                all_price_other_seller = {}
                for info_other_product in others_offer:
                    if (
                        info_other_product["seller_name"].lower().replace(" ", "")
                        == "cdiscount"
                    ):
                        price_site_seller = float(
                            info_other_product["seller_price"]
                            .replace("\n", ".")
                            .replace("€", "")
                        )
                        info_product["price_site_seller"] = price_site_seller
                    else:
                        all_price_other_seller[
                            info_other_product["seller_name"]
                        ] = float(
                            info_other_product["seller_price"]
                            .replace("\n", ".")
                            .replace("€", "")
                        )
                info_product["ean"] = ean
                info_product["list_other_price"] = all_price_other_seller
                product_check.add(ean)
            info_all_seller_product.append(info_product)
        except Exception:
            pass

In [11]:
absolute_threshold = 50
relative_threshold = 0.3
product_to_keep = dict()
for dict_product_info in info_all_seller_product:
    try:
        list_sorted_price_other_seller = [
            v
            for k, v in sorted(
                dict_product_info["list_other_price"].items(), key=lambda x: x[1]
            )
        ]
        diff_price = (
            list_sorted_price_other_seller[0] - dict_product_info["price_site_seller"]
        )
        relative_diff = (
            list_sorted_price_other_seller[0] - dict_product_info["price_site_seller"]
        ) / dict_product_info["price_site_seller"]
        if relative_diff > relative_threshold:
            if diff_price > absolute_threshold:
                product_to_keep[str(dict_product_info["ean"])] = (
                    dict_product_info["price_site_seller"] * 1.2
                )
        continue
    except Exception:
        continue

In [12]:
product_to_keep

{'0196119451311': 599.9879999999999,
 '4711121242069': 1199.988,
 '4710886672340': 239.988,
 '0196786486111': 899.9879999999999,
 '0196786486081': 359.988,
 '0753759232726': 455.988,
 '0196786287145': 599.9879999999999,
 '4711377002103': 959.9879999999999}

In [43]:
# for ele in product_without_concurrent:
#     print(
#         "title",
#         ele["title"],
#         "price",
#         ele["seller_price"],
#         "ean",
#         ele["ean"],
#         "name",
#         ele["seller_name_fp"],
#     )
#     print("\n")